# Exp 5 - Earliest Deadline First (EDF) Scheduling using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Simulate Earliest Deadline First scheduling for periodic real-time tasks.

EDF is a dynamic-priority scheduling policy: among ready jobs, the job with the earliest absolute deadline runs first. Priorities can change at runtime as new jobs arrive.

## Textbook Notes and Case Studies

### 1. Textbook Background

Earliest Deadline First is a dynamic-priority scheduling algorithm. At each scheduling decision, the ready task with the earliest absolute deadline receives the processor. Unlike RMS, priorities are not fixed; they change as deadlines approach.

EDF is theoretically optimal for preemptive scheduling of independent tasks on a single processor under standard assumptions. This means if any algorithm can schedule such a task set, EDF can schedule it. In practice, engineering limits still matter: context-switch overhead, blocking, interrupt latency, shared resources, and inaccurate execution-time estimates can break the ideal model.

### 2. Architecture Notes

```
Task Releases -> Compute Absolute Deadlines -> Ready Queue by Deadline
       |                    |                         |
       v                    v                         v
Release Time         r + relative_deadline       Earliest Deadline Runs
                                                        |
                                                        v
                                                Deadline Verification
```

EDF is useful when tasks have different or changing deadlines. The scheduler must continuously compare absolute deadlines, which can be slightly more complex than a fixed-priority table.

### 3. Important Formulas

Absolute deadline:

```
d_i = r_i + D_i
```

EDF selection rule:

```
run task k where d_k = min(d_i for all ready tasks)
```

For periodic independent tasks under the ideal uniprocessor EDF model, a common utilization condition is:

```
sum(C_i / T_i) <= 1
```

This condition depends on the model. If deadlines are shorter than periods or tasks share locked resources, more detailed analysis is required.

### 4. Classroom Case Studies

Case Study A - Mixed Sensor Fusion:
Camera, radar, and ultrasonic data may arrive with different freshness requirements. EDF can prioritize whichever data item is closest to losing usefulness.

Case Study B - Emergency Event Handling:
If a sudden obstacle event creates a short deadline task, EDF immediately ranks it above less urgent work. This is useful for event-driven real-time systems.

Case Study C - Compute Overload:
When total demanded utilization exceeds available CPU time, EDF still misses deadlines. The first misses usually occur near periods of dense deadline clustering. This demonstrates that a scheduler cannot solve overload alone.

### 5. Analysis Checklist

For every task, record release time, computation time, relative deadline, absolute deadline, execution interval, and deadline result. Explain priority changes over time. Compare EDF with RMS only after checking whether both algorithms are being tested under the same task model.

### 6. Source Notes

- EDF theory is closely related to the real-time scheduling foundations in Liu and Layland, 1973: https://dl.acm.org/doi/10.1145/321738.321743
- Python simulation timing support: https://docs.python.org/3/library/time.html


## Architecture

```text
Periodic Task Releases
          |
          v
Ready Queue
  |-- job name
  |-- remaining execution time
  |-- absolute deadline
          |
          v
EDF Dispatcher
  |-- sort ready jobs by earliest deadline
  |-- execute selected job
          |
          v
Deadline Monitor
```

## Formulas and Required Theory

Absolute deadline:

\[
D_i^{abs} = release_i + D_i^{rel}
\]

Processor utilization:

\[
U = \sum_{i=1}^{n}\frac{C_i}{T_i}
\]

For independent periodic tasks with deadlines equal to periods on one processor, a common EDF feasibility condition is:

\[
U \le 1
\]

This notebook uses that utilization check as a lab-level feasibility test and also simulates a schedule timeline.

## In-Lab Method

1. Define task periods, execution times, and deadlines.
2. Generate jobs over the hyperperiod.
3. At each time slot, select the ready job with the earliest absolute deadline.
4. Execute one time unit.
5. Count deadline misses.

In [1]:
from math import gcd
from functools import reduce

def lcm(a, b):
    return a * b // gcd(a, b)

tasks = [
    {"name": "Sensor", "period": 4, "exec": 1, "deadline": 4},
    {"name": "Fusion", "period": 6, "exec": 2, "deadline": 6},
    {"name": "Control", "period": 12, "exec": 3, "deadline": 12},
]
hyperperiod = reduce(lcm, [t["period"] for t in tasks])
ready = []
timeline = []
misses = 0
for time in range(hyperperiod):
    for task in tasks:
        if time % task["period"] == 0:
            ready.append({"name": task["name"], "remaining": task["exec"], "deadline": time + task["deadline"]})
    ready.sort(key=lambda job: job["deadline"])
    if ready:
        job = ready[0]
        timeline.append(job["name"])
        job["remaining"] -= 1
        if job["remaining"] == 0:
            ready.pop(0)
    else:
        timeline.append("Idle")
    late = [job for job in ready if time + 1 > job["deadline"]]
    misses += len(late)
    ready = [job for job in ready if time + 1 <= job["deadline"]]

print("EXP 5 - IN-LAB EDF RESULT")
print("Hyperperiod:", hyperperiod)
print("Processor utilization:", round(sum(t["exec"] / t["period"] for t in tasks), 3))
print("Deadline misses:", misses)
print("Timeline:", " | ".join(timeline))

EXP 5 - IN-LAB EDF RESULT
Hyperperiod: 12
Processor utilization: 0.833
Deadline misses: 0
Timeline: Sensor | Fusion | Fusion | Control | Sensor | Control | Control | Fusion | Fusion | Sensor | Idle | Idle


## Post-Lab Method

The post-lab cell compares normal, near-limit, and overload task sets using the EDF utilization rule.

In [2]:
def edf_feasibility(task_set):
    return sum(c / p for c, p in task_set) <= 1.0

sets = {
    "Normal": [(1, 4), (2, 6), (3, 12)],
    "Near limit": [(2, 5), (2, 6), (1, 10)],
    "Overload": [(2, 4), (3, 6), (4, 10)],
}
print("EXP 5 - POST-LAB EDF FEASIBILITY")
print(f"{'Case':12} {'Utilization':>12} {'EDF feasible':>14}")
for name, pairs in sets.items():
    util = sum(c / p for c, p in pairs)
    print(f"{name:12} {util:12.3f} {str(edf_feasibility(pairs)):>14}")

EXP 5 - POST-LAB EDF FEASIBILITY
Case          Utilization   EDF feasible
Normal              0.833           True
Near limit          0.833           True
Overload            1.400          False


## What to Write in the Lab Record

- Show the task set.
- Show total utilization.
- Include the EDF timeline.
- Explain how EDF differs from RMS: EDF priority is dynamic, RMS priority is fixed.

## References

- EDF scheduling overview: https://en.wikipedia.org/wiki/Earliest_deadline_first_scheduling
- Real-time scheduling lecture note, EDF assigns earlier deadlines higher priority: https://os.inf.tu-dresden.de/